# Kinetic-EFIT End-to-End Chain

This notebook drives the full VEST kinetic pipeline over the consolidated
`vaft.code.kineticEfit` API for a single equilibrium slice:

1. **Load** a magnetic-EFIT `gfile` and the raw Thomson-scattering / charge-exchange
   diagnostics into an ODS.
2. **`build_kinetic_core_profiles`** — psi_N mapping + `ne`/`Te`/`Ti`/`Vtor` fits +
   `core_profiles` with `pressure_thermal`, `p = e·nₑ·(Tₑ+Tᵢ)` (pure ODS→ODS).
3. **`kinetic_pressure_points`** — the `RPRESS`/`PRESSR`/`SIGPRE`/`FWTPRE` constraint
   points (raw-6pt encoding) that feed the kinetic EFIT.
4. **`run_kinetic_efit`** — kinetic-pressure EFIT with a `PLASMA`-scale sweep
   (needs `$EFIT`).
5. **`refine_equilibrium`** — CHEASE refinement of the best equilibrium
   (needs `$CHEASE`).
6. **Compare** the magnetic, kinetic, and CHEASE-refined equilibria.

Each external-code stage degrades gracefully: if `$EFIT` / `$CHEASE` is unset the
notebook prints a clear `skipped: …` message and the remaining cells stay
executable.

In [ ]:
import os
from pathlib import Path
import tempfile

import matplotlib.pyplot as plt
import numpy as np

import vaft
from vaft.data.resources import data_path
from vaft.data import read_geqdsk
from vaft.code.kineticEfit import (
    build_kinetic_core_profiles,
    kinetic_pressure_points,
    KineticEFITConfig,
    prepare_kinetic_efit_inputs,
    run_kinetic_efit,
)
from vaft.code.chease import CHEASEConfig, find_chease_executable, refine_equilibrium

vaft.apply_omfit_compat_patches()
from omfit_classes.omfit_eqdsk import OMFITgeqdsk

# --- slice selection (48224 @ 300 ms is the shipped known-good converging slice) ---
SHOT = 48224
TIME_MS = 300
ION_SOURCE = "auto"        # auto -> prefer IDS_<shot>.mat, else CES_<shot>.mat

DATA_ROOT = data_path()    # packaged vaft/data root
WORKROOT = Path(tempfile.mkdtemp(prefix=f"vaft-kinetic-{SHOT}-{TIME_MS}-"))


def _resolve_gfile(shot, time_ms):
    root = DATA_ROOT / "efit_scaled_f" / str(shot)
    cand = sorted(root.glob(f"g*.{time_ms:05d}"))
    if not cand:
        raise FileNotFoundError(f"No gfile for {shot}@{time_ms}ms under {root}")
    return cand[0]


def _resolve_ion_source(shot, source):
    if source == "ids" or (source == "auto" and (DATA_ROOT / f"IDS_{shot}.mat").exists()):
        return "ids", None                       # IDS_<shot>.mat resolved by the loader
    ces = DATA_ROOT / f"CES_{shot}.mat"
    if ces.exists():
        return "ces", str(ces)
    raise FileNotFoundError(f"No IDS_{shot}.mat or CES_{shot}.mat for shot {shot}")


# Binaries are resolved from env vars (graceful skip when unset).
EFIT_EXE = os.environ.get("EFIT")                # KineticEFITConfig also falls back to $EFIT
_chease_found = find_chease_executable()
CHEASE_EXE = os.environ.get("CHEASE") or (str(_chease_found) if _chease_found else None)

print(f"Slice      : {SHOT} @ {TIME_MS} ms")
print(f"Data root  : {DATA_ROOT}")
print(f"Work dir   : {WORKROOT}")
print(f"$EFIT      : {EFIT_EXE or '(unset -> kinetic EFIT will be skipped)'}")
print(f"CHEASE exe : {CHEASE_EXE or '(unset -> CHEASE refine will be skipped)'}")

## 1. Load The Magnetic Equilibrium And Raw Kinetic Diagnostics

The magnetic-EFIT `gfile` seeds the ODS `equilibrium`; Thomson scattering supplies
`nₑ`/`Tₑ` and the ion Doppler (IDS/CES) supplies `Tᵢ`/`Vtor`. This cell only performs
I/O — no profile physics yet. Data files live under the packaged `vaft/data`
directory (`efit_scaled_f/<shot>/` for gfiles, `IDS_<shot>.mat` / `CES_<shot>.mat`
for the ion diagnostic).

In [ ]:
gfile_path = _resolve_gfile(SHOT, TIME_MS)
geq = OMFITgeqdsk(str(gfile_path))
geq["fluxSurfaces"].load()

ods = geq.to_omas()
ods["equilibrium.ids_properties.homogeneous_time"] = 1
vaft.machine_mapping.dataset_description(
    ods, source=SHOT,
    options={"source_type": "shot",
             "description": f"VEST kinetic chain {SHOT}@{TIME_MS}ms (TS + ion Doppler)"},
)

# electrons: Thomson scattering (NeTe_<shot>.mat native 7-channel schema)
vaft.machine_mapping.thomson_scattering(ods, SHOT)
# ions: IDS or CES -> charge_exchange IDS
ion_opt, ce_mat = _resolve_ion_source(SHOT, ION_SOURCE)
vaft.machine_mapping.charge_exchange(ods, shotnumber=SHOT, options=ion_opt, mat_file=ce_mat)

print(f"gfile        : {gfile_path.name}")
print(f"Ip (magnetic): {float(geq['CURRENT']):.6e} A")
print(f"B0           : {float(geq['BCENTR']):.4f} T")
print(f"ion source   : {ion_opt}")

## 2. Build The Kinetic Core Profiles

`build_kinetic_core_profiles` is a pure ODS→ODS orchestration of the
`vaft.process.profile` chain: it maps TS and CX data onto the equilibrium `psi_N`,
fits `nₑ`/`Tₑ` (from TS) and `Tᵢ`/`Vtor` (from the ion diagnostic), and writes
`core_profiles.profiles_1d[0]` including `pressure_thermal`
(`p = e·nₑ·(Tₑ+Tᵢ)`, with `n_i ≈ n_e`). The fit modes are passed through to the
underlying fitters. We then read the fitted profiles back off the ODS and plot
`nₑ / Tₑ / Tᵢ / Vtor / pressure_thermal` on the equilibrium `rho_tor_norm` grid.

In [ ]:
ods = build_kinetic_core_profiles(
    ods, geq, TIME_MS,
    te_mode="polynomial",
    ne_mode="free_exponential",
    ti_mode="polynomial",
    vtor_mode="polynomial",
    ion_index=0,
)

cp = "core_profiles.profiles_1d.0"
rho = np.asarray(ods[f"{cp}.grid.rho_tor_norm"])
ne = np.asarray(ods[f"{cp}.electrons.density"])
te = np.asarray(ods[f"{cp}.electrons.temperature"])
ti = np.asarray(ods[f"{cp}.ion.0.temperature"])
vtor = np.asarray(ods[f"{cp}.ion.0.velocity.toroidal"])
pth = np.asarray(ods[f"{cp}.pressure_thermal"])

print(f"grid points : {rho.size}")
print(f"ne0         : {ne[0]:.3e} m^-3")
print(f"Te0         : {te[0]:.1f} eV")
print(f"Ti0         : {ti[0]:.1f} eV")
print(f"Vtor0       : {vtor[0]:.3e} m/s")
print(f"p_th0       : {pth[0]:.1f} Pa   (expect ~132 Pa for 48224@300)")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
panels = [
    (ne,   r"$n_e$ [m$^{-3}$]", "tab:blue"),
    (te,   r"$T_e$ [eV]",       "tab:red"),
    (ti,   r"$T_i$ [eV]",       "tab:orange"),
    (vtor, r"$V_{tor}$ [m/s]",  "tab:green"),
    (pth,  r"$p_{thermal}$ [Pa]", "tab:purple"),
]
for ax, (y, ylabel, color) in zip(axes.flat, panels):
    ax.plot(rho, y, color=color, lw=2)
    ax.set_xlabel(r"$\rho_{tor,N}$")
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.25)
axes.flat[-1].axis("off")   # 6th panel unused (5 profiles)
fig.suptitle(f"Kinetic core profiles   {SHOT} @ {TIME_MS} ms", fontsize=13)
fig.tight_layout()
plt.show()

## 3. Build The Kinetic-Pressure Constraint Points

`kinetic_pressure_points` turns the fitted profiles into the EFIT pressure
constraint. With `encoding='raw6'` it emits the 5 Thomson points at their real
major radius `R>0` plus a 6th `psi_N=1, p=0` edge anchor (`RPRESS=-1.0`). Each
point carries `PRESSR = e·nₑ·(Tₑ+Tᵢ)` and the propagated error
`SIGPRE = e·√((Tₑ+Tᵢ)²σ_nₑ² + nₑ²σ_Tₑ² + nₑ²σ_Tᵢ²)` floored at `sigma_floor·p`.

In [ ]:
points = kinetic_pressure_points(ods, TIME_MS, geq=geq, encoding="raw6")

print(f"encoding : raw6  ({len(points.rpress)} points)")
print(f"{'RPRESS':>10}{'PRESSR[Pa]':>14}{'SIGPRE[Pa]':>14}{'FWTPRE':>10}")
for i in range(len(points.rpress)):
    print(f"{points.rpress[i]:>10.4f}{points.pressr[i]:>14.3f}"
          f"{points.sigpre[i]:>14.3f}{points.fwtpre[i]:>10.3f}")

rp = np.asarray(points.rpress, dtype=float)
pr = np.asarray(points.pressr, dtype=float)
sg = np.asarray(points.sigpre, dtype=float)
real = rp > 0.0                                   # raw6: real-R points; anchor is RPRESS=-1.0

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.errorbar(rp[real], pr[real], yerr=sg[real], fmt="o", color="tab:purple",
            capsize=4, label="TS kinetic pressure")
ax.set_xlabel("R [m]")
ax.set_ylabel(r"$p = e\,n_e\,(T_e + T_i)$  [Pa]")
ax.set_title(f"Kinetic-pressure points  ({int(real.sum())} at real R + edge anchor)")
ax.grid(True, alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

## 4. Run The Kinetic-Pressure EFIT

`run_kinetic_efit` injects the pressure constraint into the base magnetic kfile,
runs EFIT across the `PLASMA`-scale sweep, and keeps the largest scale that
converges. The executable is taken from `KineticEFITConfig.executable` or `$EFIT`;
when neither resolves, the result comes back with `status='skipped'` instead of an
error. This cell prints a clear `skipped:` line if `$EFIT` is unset.

In [ ]:
efit_config = KineticEFITConfig(
    executable=EFIT_EXE,          # None -> resolves $EFIT; still None -> status='skipped'
    workdir=str(WORKROOT / "kinetic_efit"),
    shot=SHOT,
    time_ms=TIME_MS,
    encoding="raw6",
    timeout=600,
)

efit_result = None
if EFIT_EXE is None:
    print("skipped: set $EFIT (or KineticEFITConfig.executable) to run the kinetic-pressure EFIT.")
else:
    try:
        efit_inputs = prepare_kinetic_efit_inputs(ods, geq, efit_config)
        efit_result = run_kinetic_efit(efit_inputs, efit_config)
        if efit_result.status == "skipped":
            print(f"skipped: {efit_result.reason or 'EFIT executable unresolved (set $EFIT).'}")
        elif efit_result.ok and efit_result.converged:
            print(f"converged at PLASMA scale = {efit_result.scale}")
            print(f"chi2  : {efit_result.chi2}")
            print(f"gfile : {efit_result.gfile}")
        else:
            print(f"did not converge (status={efit_result.status}): {efit_result.reason}")
    except Exception as exc:
        print(f"skipped: kinetic EFIT raised {type(exc).__name__}: {exc}")

## 5. Refine The Equilibrium With CHEASE

`refine_equilibrium` runs the CHEASE inverse solver on the best available
equilibrium — the kinetic-EFIT `gfile` if that step converged, otherwise the
magnetic `gfile`. CHEASE is discovered from `$CHEASE` / `CHEASE_EXEC_DIR` /
common paths; the cell prints a clear `skipped:` line when no executable is found.
For 48224@300 the shipped bundle expects `|Ip|≈137.6 kA`, `q0≈1.86`, `p0≈132 Pa`.

In [ ]:
# refine the best available equilibrium: kinetic gfile if EFIT converged, else magnetic
if efit_result is not None and getattr(efit_result, "gfile", None):
    chease_source = efit_result.gfile
    src_label = "kinetic-EFIT gfile"
else:
    chease_source = gfile_path
    src_label = "magnetic gfile (kinetic EFIT unavailable)"

chease_result = None
if CHEASE_EXE is None:
    print("skipped: set $CHEASE (or CHEASE_EXEC_DIR) to run the CHEASE refinement.")
else:
    try:
        chease_config = CHEASEConfig(
            executable=str(CHEASE_EXE),
            workdir=str(WORKROOT / "chease"),
            create_plot=False,
            cleanup=False,
            timeout=300,
        )
        print(f"refining {src_label}: {Path(str(chease_source)).name}")
        chease_result = refine_equilibrium(str(chease_source), chease_config)
        if chease_result.ok and chease_result.refined_geqdsk:
            print(f"refined gfile : {chease_result.refined_geqdsk}")
            for key, value in chease_result.comparison.items():
                print(f"  {key}: {value:.6g}")
        else:
            print("CHEASE produced no refined GEQDSK in this environment.")
            tail = (chease_result.stderr or "").strip().splitlines()[-8:]
            if tail:
                print("\n".join(tail))
    except Exception as exc:
        print(f"skipped: CHEASE raised {type(exc).__name__}: {exc}")

## 6. Compare Magnetic vs Kinetic vs CHEASE Equilibria

Overlay the safety factor `q(ψ_N)`, pressure `p(ψ_N)`, and last-closed boundary for
every equilibrium that is actually available. With no binaries only the magnetic
equilibrium is shown; adding `$EFIT` overlays the kinetic reconstruction, and
`$CHEASE` overlays the refined result.

In [ ]:
def _try_read(path):
    try:
        return read_geqdsk(str(path)) if path else None
    except Exception:
        return None


eqs = [("Magnetic EFIT", gfile_path, "tab:blue", "-")]
if efit_result is not None and getattr(efit_result, "gfile", None):
    eqs.append(("Kinetic EFIT", efit_result.gfile, "tab:red", "-"))
if chease_result is not None and getattr(chease_result, "refined_geqdsk", None):
    eqs.append(("CHEASE refined", chease_result.refined_geqdsk, "tab:green", "--"))

loaded = [(label, _try_read(path), color, ls) for label, path, color, ls in eqs]
loaded = [t for t in loaded if t[1] is not None]

fig, (ax_q, ax_p, ax_b) = plt.subplots(1, 3, figsize=(16, 5))
for label, g, color, ls in loaded:
    x = np.linspace(0.0, 1.0, int(g["NW"]))
    ax_q.plot(x, np.asarray(g["QPSI"], dtype=float), color=color, ls=ls, lw=1.8, label=label)
    ax_p.plot(x, np.asarray(g["PRES"], dtype=float), color=color, ls=ls, lw=1.8, label=label)
    rb = np.asarray(g.get("RBBBS", []), dtype=float)
    zb = np.asarray(g.get("ZBBBS", []), dtype=float)
    if rb.size and zb.size:
        n = min(rb.size, zb.size)
        ax_b.plot(rb[:n], zb[:n], color=color, ls=ls, lw=1.8, label=label)

ax_q.set_title("Safety factor q"); ax_q.set_xlabel(r"$\psi_N$"); ax_q.grid(True, alpha=0.25); ax_q.legend()
ax_p.set_title("Pressure [Pa]"); ax_p.set_xlabel(r"$\psi_N$"); ax_p.grid(True, alpha=0.25); ax_p.legend()
ax_b.set_title("Boundary"); ax_b.set_xlabel("R [m]"); ax_b.set_ylabel("Z [m]")
ax_b.set_aspect("equal", adjustable="box"); ax_b.grid(True, alpha=0.25); ax_b.legend()
fig.suptitle(f"Equilibrium comparison   {SHOT} @ {TIME_MS} ms", fontsize=13)
fig.tight_layout()
plt.show()

print(f"Equilibria compared: {[label for label, *_ in loaded]}")
if len(loaded) == 1:
    print("Only the magnetic equilibrium is available - set $EFIT and $CHEASE "
          "to populate the kinetic and refined overlays.")

print("\nOne-call equivalent of this whole notebook:")
print("  from vaft.code.kineticEfit import run_kinetic_chain")
print("  out = run_kinetic_chain(ods, geq, TIME_MS, efit_config=efit_config,")
print("                          chease_config=chease_config)")
print("  # -> {'ods', 'kinetic_efit': KineticEFITResult, 'chease': CHEASEResult|None}")